# 01. Inspect Replogle Data and Model Inputs

This notebook inspects the processed Replogle `.h5ad` file and connects its contents to the tensors used by PerturbDiff.

- Each row corresponds to one single cell.
- `obs` stores covariates such as perturbation gene, batch, and cell line.
- `obsm['X_hvg']` stores the 2,000 HVG expression space used by the Replogle setup.
- During training and sampling, cells are grouped into cell sets, so the common tensor shape is `[batch, cell_set_size, gene_dim]`.

In [2]:
from pathlib import Path
import pickle

import anndata as ad
import numpy as np
import pandas as pd

# Set this to the PerturbDiff repository root.
# If the notebook server is launched from the repo root, the default is already correct.
PROJECT_ROOT = Path("/home/lwf/PerturbDiff").resolve()
assert (PROJECT_ROOT / "src" / "apps" / "run" / "rawdata_diffusion_sampling.py").exists(), \
    "Set PROJECT_ROOT to the PerturbDiff repository root before continuing."
DATASET_NAME = "replogle"
DATA_ROOT = PROJECT_ROOT / "data" / DATASET_NAME
PERTURB_ROOT = DATA_ROOT / "perturb_data"
H5AD_PATH = PERTURB_ROOT / "finetune_data" / "nadig_processed_data" / "replogle.h5ad"
SELECTED_GENE_PATH = PERTURB_ROOT / "selected_genes" / "replogle_real_selected_genes.pkl"

print("H5AD_PATH:", H5AD_PATH)
print("SELECTED_GENE_PATH:", SELECTED_GENE_PATH)

H5AD_PATH: /home/lwf/PerturbDiff/data/replogle/perturb_data/finetune_data/nadig_processed_data/replogle.h5ad
SELECTED_GENE_PATH: /home/lwf/PerturbDiff/data/replogle/perturb_data/selected_genes/replogle_real_selected_genes.pkl


## 1. Open the h5ad File in Read-Only Mode

`backed='r'` avoids loading the full expression matrix into memory. This is useful for first checking metadata and array shapes.

In [ ]:
adata = ad.read_h5ad(H5AD_PATH, backed="r")

print("adata shape:", adata.shape)  # (num_cells, num_genes)
print("obs columns:", list(adata.obs.columns))    # cell level metadata
print("var columns:", list(adata.var.columns))   # gene level metadata
print("obsm keys:", list(adata.obsm.keys()))
print("layers:", list(adata.layers.keys()))

adata shape: (643413, 6642)
obs columns: ['gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line']
var columns: ['highly_variable', 'means', 'dispersions', 'dispersions_norm']
obsm keys: ['X_hvg']
layers: []


In [5]:
adata.obs

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line
cell_barcode,,,,,,,,,,
AAACCCAAGAATAGTC-3,3,KIAA1143,ENSG00000163807,P1P2,4360_KIAA1143_P1P2_ENSG00000163807,KIAA1143_+_44803075.23-P1P2|KIAA1143_+_4480308...,0.114029,11234.0,-0.611091,hepg2
AAACCCAAGAGGTATT-55,55,SEPHS2,ENSG00000179918,P1P2,7779_SEPHS2_P1P2_ENSG00000179918,SEPHS2_-_30457178.23-P1P2|SEPHS2_+_30457164.23...,0.165242,45146.0,0.208833,hepg2
AAACCCAAGAGTGACC-39,39,POLR2H,ENSG00000163882,P1P2,6549_POLR2H_P1P2_ENSG00000163882,POLR2H_+_184081227.23-P1P2|POLR2H_-_184081237....,0.162209,20190.0,-0.025114,hepg2
AAACCCAAGATGGCAC-43,43,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.071596,23912.0,2.079665,hepg2
AAACCCAAGCAACAAT-16,16,TMEM214,ENSG00000119777,P1P2,9042_TMEM214_P1P2_ENSG00000119777,TMEM214_-_27255873.23-P1P2|TMEM214_-_27255853....,0.109515,8282.0,-0.214576,hepg2
...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCTGATGGT-14,14,SLC1A5,ENSG00000105281,P1,7976_SLC1A5_P1_ENSG00000105281,SLC1A5_+_47291839.23-P1|SLC1A5_+_47291829.23-P1,0.062634,15423.0,0.347911,rpe1
TTTGTTGTCTGCACCT-44,44,MAX,ENSG00000125952,P1P2,4871_MAX_P1P2_ENSG00000125952,MAX_+_65569008.23-P1P2|MAX_-_65568906.23-P1P2,0.079629,22228.0,0.767826,rpe1
TTTGTTGTCTGGGCAC-32,32,ATP6V0C,ENSG00000185883,P1P2,675_ATP6V0C_P1P2_ENSG00000185883,ATP6V0C_+_2564168.23-P1P2|ATP6V0C_-_2563995.23...,0.049527,12377.0,-0.004215,rpe1


`adata.obs` 中的

```python
['gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript',
 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'cell_line']
```

可以按三类理解：扰动标签、实验批次/QC、细胞来源。

| column | 含义 |
|---|---|
| `gem_group` | 10x Genomics 的 GEM group / sequencing batch / library batch。PerturbDiff 把它当作 batch covariate，用来建模批次差异。 |
| `gene` | 这个 cell 对应的 perturbation target gene，也就是被 CRISPR 扰动的基因名。control cell 通常是 `non-targeting`。这是最重要的扰动标签。 |
| `gene_id` | `gene` 对应的基因 ID，通常可能是 Ensembl ID 或数据处理时保留的 target gene identifier。比 `gene` 更偏数据库编号。 |
| `transcript` | 被 guide / perturbation annotation 对应到的 transcript ID 或 transcript-level target 信息。 |
| `gene_transcript` | `gene` 和 `transcript` 的组合字段，用来唯一标识 gene-transcript target。 |
| `sgID_AB` | sgRNA / guide RNA 的 ID。`AB` 通常表示 guide assignment 的组合标签，可能包含一个或多个 guide 的 assignment 信息。它比 `gene` 更底层，因为多个 sgRNA 可以靶向同一个 gene。 |
| `mitopercent` | 每个 cell 中 mitochondrial reads / UMIs 的比例。常用于单细胞 QC，比例过高可能表示细胞质量差或死亡细胞。 |
| `UMI_count` | 每个 cell 的总 UMI 数，也就是 library size / sequencing depth。 |
| `z_gemgroup_UMI` | 在每个 `gem_group` 内对 `UMI_count` 做 z-score 标准化后的值。用来衡量某个 cell 在同一 batch 内的 UMI_count 是否异常。 |
| `cell_line` | 这个 cell 来自哪个 cell line，例如 `hepg2` 等。PerturbDiff 把它作为 cell type / cell line covariate。 |

对 PerturbDiff 当前 Replogle 配置来说，最核心的是这三个：

```python
gene
gem_group
cell_line
```

对应配置里：

```yaml
pert_col: gene
perturbseq_batch_col: gem_group
cell_line_key: cell_line
control_pert: non-targeting
```

其他列主要是更细的扰动注释或 QC 信息：

```text
sgID_AB, gene_id, transcript, gene_transcript  -> perturbation annotation
mitopercent, UMI_count, z_gemgroup_UMI         -> cell quality / sequencing depth QC
```

所以简单来说：

```text
gene tells the model what perturbation was applied.
gem_group tells the model which experimental batch the cell came from.
cell_line tells the model the cellular context.
X_hvg provides the 2,000-dimensional expression vector used as model input.
```

In [4]:
adata.var

,highly_variable,means,dispersions,dispersions_norm
gene_name_index,,,,
NOC2L,False,0.690140,0.089601,-0.625512
HES4,True,0.693763,1.156615,3.630171
ISG15,True,0.756257,1.104892,3.423880
SDF4,False,0.706835,0.167762,-0.313772
B3GALT6,False,0.283483,0.041215,-0.237526
...,...,...,...,...
MT-ND4L,True,1.379477,0.915507,0.915423
MT-ND4,True,4.453991,2.885937,0.192740
MT-ND5,True,2.873720,1.403472,0.412011


## 2. Check the Columns Used by the PerturbDiff Config

The Replogle config expects the following fields:
- `gene`: perturbation gene.
- `gem_group`: Perturb-seq batch.
- `cell_line`: cell line condition.
- `non-targeting`: control perturbation label.

In [3]:
required_obs = ["gene", "gem_group", "cell_line"]
missing_obs = [c for c in required_obs if c not in adata.obs.columns]
if missing_obs:
    raise KeyError("Missing required obs columns: " + ", ".join(missing_obs))

summary = {}
for col in required_obs:
    values = adata.obs[col]
    summary[col] = {
        "n_unique": values.nunique(),
        "examples": list(values.astype(str).unique()[:8]),
    }

pd.DataFrame(summary).T

,n_unique,examples
gene,2024,"[KIAA1143, SEPHS2, POLR2H, TFAM, TMEM214, BAP1..."
gem_group,56,"[3, 55, 39, 43, 16, 14, 47, 12]"
cell_line,4,"[hepg2, jurkat, k562, rpe1]"


In [4]:
control_mask = adata.obs["gene"].astype(str) == "non-targeting"
hepg2_mask = adata.obs["cell_line"].astype(str) == "hepg2"

print("control cells:", int(control_mask.sum()))
print("hepg2 cells:", int(hepg2_mask.sum()))
print("hepg2 non-targeting cells:", int((control_mask & hepg2_mask).sum()))

control cells: 39165
hepg2 cells: 96616
hepg2 non-targeting cells: 4976


## 3. Check the 2,000 HVG Input Space

The Replogle scratch setting uses `X_hvg` and sets the model input dimension to `2000`. This cell verifies that `obsm['X_hvg']` exists and has the expected dimensionality.

In [5]:
if "X_hvg" not in adata.obsm:
    raise KeyError("obsm['X_hvg'] not found. This Replogle file may not be the processed PerturbDiff version.")

x_hvg = adata.obsm["X_hvg"]
print("X_hvg shape:", x_hvg.shape)
assert x_hvg.shape[1] == 2000, "Expected 2,000 HVG features for the Replogle scratch setup."

X_hvg shape: (643413, 2000)


## 4. Load the Selected Genes

The selected gene list defines the gene order of the model output space. The model predicts values for this fixed set of 2,000 genes.

In [6]:
with open(SELECTED_GENE_PATH, "rb") as f:
    selected_genes = pickle.load(f)

print("type:", type(selected_genes))
print("n selected genes:", len(selected_genes))
print("first 20 genes:", list(selected_genes)[:20])

type: <class 'list'>
n selected genes: 2000
first 20 genes: ['HES4', 'ISG15', 'MIB2', 'CEP104', 'RERE', 'ENO1', 'SLC25A33', 'CTNNBIP1', 'KIF1B', 'KIAA2013', 'PRDM2', 'SPEN', 'NBPF1', 'RCC2', 'CAMK2N1', 'DDOST', 'RPL11', 'LYPLA2', 'GALE', 'HMGCL']


## 5. Build a Small Cell-Set Tensor

This is a lightweight tensor construction for inspecting the shape. The training dataloader applies the full grouping logic using perturbation, cell line, batch, and split metadata.

In [7]:
cell_set_size = 8
demo_gene = adata.obs.loc[~control_mask, "gene"].astype(str).iloc[0]
demo_mask = adata.obs["gene"].astype(str) == demo_gene
demo_indices = np.flatnonzero(np.asarray(demo_mask))[:cell_set_size]

demo_x = np.asarray(adata.obsm["X_hvg"][demo_indices])
demo_x = demo_x.reshape(1, cell_set_size, 2000)

print("demo perturbation:", demo_gene)
print("cell set tensor shape:", demo_x.shape)
print("meaning: [one perturbation condition, cells in the set, 2,000 HVG features]")

demo perturbation: KIAA1143
cell set tensor shape: (1, 8, 2000)
meaning: [one perturbation condition, cells in the set, 2,000 HVG features]
